In [ ]:
#setup
import math
import statistics as st
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
pd.options.display.precision = 4
rng = np.random.default_rng(42)
print(sys.executable)

In [ ]:
#same mean, very different data
class_a = np.array([60, 62, 64, 66, 68])
class_b = np.array([40, 55, 65, 75, 85])
print("means :", class_a.mean(), class_b.mean())
print("ranges:", np.ptp(class_a), np.ptp(class_b))            # np.ptp = max - min

fig, ax = plt.subplots(figsize=(8, 2.4))
ax.plot(class_a, [1] * len(class_a), "o", ms=10, label="Class A")
ax.plot(class_b, [2] * len(class_b), "s", ms=10, label="Class B")
ax.axvline(64, color="gray", ls="--")
ax.set_yticks([1, 2])
ax.set_yticklabels(["A", "B"])
ax.set_xlabel("marks")
plt.show()

In [ ]:
#four methods
def quartiles_halves(data):
    """Textbook method: split the sorted data in half, leaving the median out when n is odd."""
    s = sorted(data)
    n = len(s)
    lower, upper = s[: n // 2], s[n // 2 + n % 2:]
    return st.median(lower), st.median(s), st.median(upper)

def compare_methods(data):
    data = list(data)
    return pd.DataFrame({
        "numpy linear (default)": np.percentile(data, [25, 50, 75]),
        "statistics exclusive (default)": st.quantiles(data, n=4),
        "statistics inclusive": st.quantiles(data, n=4, method="inclusive"),
        "halves, median left out": quartiles_halves(data),
    }, index=["Q1", "Q2", "Q3"]).T

samples = {
    "1 to 9": range(1, 10),
    "class A marks": [3, 4, 5, 5, 6, 7, 8, 8, 8, 9],
    "2 4 4 4 5 5 7 9": [2, 4, 4, 4, 5, 5, 7, 9],
}
for name, data in samples.items():
    print(name)
    print(compare_methods(data), "\n")

In [ ]:
#percentiles in practice
scores = np.array([55, 60, 62, 65, 70, 72, 78, 80, 85, 92])

print(np.percentile(scores, [10, 25, 50, 75, 90]))             # percentages 0 to 100
print(np.quantile(scores, [0.10, 0.25, 0.50, 0.75, 0.90]))     # fractions 0 to 1: the same numbers
print(pd.Series(scores).quantile([0.25, 0.5, 0.75]).tolist())

# What percentile is a score of 78? The answer depends on how you count ties
for kind in ("strict", "weak", "mean"):
    print(f"{kind:<7}", stats.percentileofscore(scores, 78, kind=kind))

In [ ]:
#the five-number summary
data_out = np.array([12, 13, 14, 15, 15, 16, 17, 18, 19, 55])      # one suspicious value

def five_number_summary(data):
    q1, med, q3 = np.percentile(data, [25, 50, 75])
    return {"min": np.min(data), "Q1": q1, "median": med, "Q3": q3,
            "max": np.max(data), "IQR": q3 - q1}

print(five_number_summary(data_out))
print(pd.Series(data_out).describe())                    # count, mean, std, min, 25%, 50%, 75%, max

In [ ]:
#finding outliers and drawing the box plot
def iqr_outliers(data, k=1.5):
    data = np.asarray(data)
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    return data[(data < low) | (data > high)], (low, high)

found, (low, high) = iqr_outliers(data_out)
print(f"fences: {low:.2f} to {high:.2f}   outliers: {found}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].boxplot(data_out, whis=1.5)
axes[0].set_title("Box plot")
axes[1].hist(data_out, bins=12, color="lightsteelblue", edgecolor="white")
for q, colour in zip(np.percentile(data_out, [25, 50, 75]), ("seagreen", "tomato", "seagreen")):
    axes[1].axvline(q, color=colour, lw=2)
axes[1].set_title("Histogram with Q1, median, Q3")
plt.tight_layout()
plt.show()

clean = data_out[data_out != 55]
print(f"mean   with 55: {data_out.mean():.2f}   without: {clean.mean():.2f}")
print(f"median with 55: {np.median(data_out):.2f}   without: {np.median(clean):.2f}")

In [ ]:
#comparing groups with box plots
a = rng.normal(70, 6, 200)
b = rng.normal(70, 15, 200)
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.boxplot([a, b])
ax.set_xticks([1, 2])
ax.set_xticklabels(["Group A", "Group B"])
ax.set_ylabel("marks")
plt.show()

df = pd.DataFrame({"group": ["A"] * 200 + ["B"] * 200, "marks": np.concatenate([a, b])})
print(df.groupby("group")["marks"].describe())

In [ ]:
#by hand, step by step
data = np.array([2, 4, 4, 4, 5, 5, 7, 9])
mean = data.mean()
table = pd.DataFrame({"x": data, "x - mean": data - mean, "squared": (data - mean) ** 2})
print(table, "\n")

ss = table["squared"].sum()
n = len(data)
print("sum of squared deviations:", ss)                                     # 32.0
print("population variance:", ss / n, "-> sd", math.sqrt(ss / n))           # 4.0 -> 2.0
print("sample variance    :", ss / (n - 1), "-> sd", math.sqrt(ss / (n - 1)))   # 4.5714 -> 2.1381

print("shortcut check:", (data ** 2).mean() - mean ** 2)                    # E[x^2] - (E[x])^2 = 4.0

In [ ]:
#the same data, different tools, different answers
x = [2, 4, 4, 4, 5, 5, 7, 9]
print("np.var(x)                :", np.var(x))                  # 4.0       population (ddof=0)
print("np.var(x, ddof=1)        :", np.var(x, ddof=1))          # 4.5714    sample
print("pd.Series(x).var()       :", pd.Series(x).var())         # 4.5714    pandas defaults to SAMPLE
print("pd.Series(x).var(ddof=0) :", pd.Series(x).var(ddof=0))   # 4.0
print("st.pvariance / st.variance:", st.pvariance(x), "/", st.variance(x))
print("st.pstdev / st.stdev     :", st.pstdev(x), "/", round(st.stdev(x), 4))
print("np.std / pd .std         :", np.std(x), "/", round(pd.Series(x).std(), 4))

In [ ]:
#seeing the correction work
true_sigma = 10                                              # true variance is 100
samples = rng.normal(0, true_sigma, size=(200_000, 5))       # 200,000 samples, each of size 5

print("true variance             :", true_sigma ** 2)
print("average of ddof=0 variance:", samples.var(axis=1, ddof=0).mean().round(2))   # about 80
print("average of ddof=1 variance:", samples.var(axis=1, ddof=1).mean().round(2))   # about 100
print("average of s (ddof=1)     :", samples.std(axis=1, ddof=1).mean().round(2))   # about 9.4

In [ ]:
#how the mean and SD respond to changes in the data
x = np.array([10, 20, 30, 40, 50])
print("original   :", x.mean(), round(x.std(ddof=1), 3), x.var(ddof=1))
print("add 100    :", (x + 100).mean(), round((x + 100).std(ddof=1), 3))       # the spread does not change
print("multiply 3 :", (x * 3).mean(), round((x * 3).std(ddof=1), 3), (x * 3).var(ddof=1))   # SD x3, variance x9

In [ ]:
#standard deviation vs mean absolute deviation, and a robust option
v = np.array([0, 0, 0, 10])
print("mean absolute deviation:", np.abs(v - v.mean()).mean())      # 3.75
print("standard deviation     :", round(v.std(), 4))                # 4.3301: squaring weights big deviations more

print("median absolute deviation:", stats.median_abs_deviation(data_out), "| SD:", round(data_out.std(ddof=1), 3))
print("IQR:", np.subtract(*np.percentile(data_out, [75, 25])))

In [ ]:
#z-scores
z = stats.zscore(data_out, ddof=1)
print(pd.DataFrame({"x": data_out, "z": z.round(2)}).tail(3))
print("largest |z|:", np.abs(z).max().round(2))                     # 2.80
print("flagged by |z| > 3:", data_out[np.abs(z) > 3])               # nothing
print("flagged by the IQR rule:", iqr_outliers(data_out)[0])        # [55]

# Standardising gives mean 0 and SD 1
standard = (data_out - data_out.mean()) / data_out.std(ddof=1)
print(round(standard.mean(), 10), round(standard.std(ddof=1), 10))

In [ ]:
#the empirical rule and Chebyshev’s inequality
normal = rng.normal(100, 15, 200_000)
m, s = normal.mean(), normal.std()
rule = {1: 0.6827, 2: 0.9545, 3: 0.9973}
for k in (1, 2, 3):
    inside = np.mean(np.abs(normal - m) <= k * s)
    print(f"normal data, within {k} SD: {inside:.4f}   (68-95-99.7 rule: {rule[k]})")

skewed = rng.exponential(10, 200_000)
m, s = skewed.mean(), skewed.std()
for k in (1.5, 2, 3):
    inside = np.mean(np.abs(skewed - m) <= k * s)
    print(f"skewed data, within {k} SD: {inside:.4f}   (Chebyshev guarantees at least {1 - 1 / k ** 2:.4f})")

In [ ]:
#coefficient of variation
heights = rng.normal(170, 10, 1000)           # cm
weights = rng.normal(70, 12, 1000)            # kg
cv = lambda a: a.std(ddof=1) / a.mean() * 100
print(f"heights: SD {heights.std(ddof=1):.1f} cm,  CV {cv(heights):.1f}%")
print(f"weights: SD {weights.std(ddof=1):.1f} kg,  CV {cv(weights):.1f}%")

In [ ]:
#one view of everything
summary = pd.DataFrame({"Class A": pd.Series(class_a), "Class B": pd.Series(class_b)}).agg(
    ["mean", "median", "min", "max", "std", "var"])
summary.loc["range"] = summary.loc["max"] - summary.loc["min"]
summary.loc["Q1"] = [np.percentile(class_a, 25), np.percentile(class_b, 25)]
summary.loc["Q3"] = [np.percentile(class_a, 75), np.percentile(class_b, 75)]
summary.loc["IQR"] = summary.loc["Q3"] - summary.loc["Q1"]
print(summary)